# Freight Rate Prediction

The goal of this project is to predict the `posted_rate` for freight loads.

The labelled development data covers January to October 2025. The final validation data contains 12,000 loads for which predictions are required.


In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

In [2]:
data = pd.read_csv('data/train-test.csv')
validation = pd.read_csv('data/validation.csv')

## Initial Data Exploration

I started by checking the shape and structure of the development data and confirming that there were no duplicate rows.

I then checked for missing values in both the development and validation data. `weight` and `market_index` had missing values. I planned to handle missing `weight` because it is used as a model feature, while `market_index` was not included in the final feature set.

I also checked the data types and converted `date` to datetime so that I could explore time-based patterns.

To better understand the data, I looked at the frequency of pickup and delivery combinations and created a month variable to examine how posted rates varied across the development period.


In [3]:
data.shape

(48000, 14)

In [4]:
data.duplicated().sum()

np.int64(0)

In [5]:
data.isna().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

In [6]:
validation.isna().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB


In [8]:
data['date'] = pd.to_datetime(data['date'])

In [9]:
summary = data.groupby(['pickup', 'delivery']).size().sort_values(ascending=False).reset_index()
summary.rename(columns= {0 : 'pickup_delivery_count'}, inplace=True)

In [10]:
summary.describe()

,pickup_delivery_count
count,4014.000000
mean,11.958146
std,7.165047
min,1.000000
25%,6.000000
50%,10.000000
75%,16.000000
max,39.000000


In [11]:
data['month'] = data['date'].dt.month

In [12]:
data.groupby(['month'])['posted_rate'].agg(['count', 'mean', 'median'])

,count,mean,median
month,,,
1,4918,2255.967048,1915.200
2,4337,2273.804801,1994.250
3,5036,2372.268092,2022.920
4,4819,2372.162308,2044.240
5,4913,2421.776342,2065.510
6,4783,2497.030115,2120.220
7,4912,2415.162030,2059.145
8,4759,2338.407661,2015.730
9,4670,2406.374013,2057.130


## Handling Missing Weight

For missing weight values, I first used the median weight for the same pickup, delivery and equipment combination.

If that combination was not available, I used the median weight for the same equipment.

The medians were calculated from the training data and then applied to the validation and future data.


In [13]:
def fill_missing_weight(train_data, data):
    lane_median = (
        train_data
        .groupby(['pickup', 'delivery', 'equipment'])['weight']
        .median()
        .reset_index(name='lane_weight_median')
    )

    equipment_median = (
        train_data
        .groupby('equipment')['weight']
        .median()
        .reset_index(name='equipment_weight_median')
    )

    data = data.copy()

    data = data.merge(
        lane_median,
        on=['pickup', 'delivery', 'equipment'],
        how='left'
    )

    data = data.merge(
        equipment_median,
        on='equipment',
        how='left'
    )

    data['weight'] = (
        data['weight']
        .fillna(data['lane_weight_median'])
        .fillna(data['equipment_weight_median'])
    )

    return data.drop(
        columns=['lane_weight_median', 'equipment_weight_median']
    )

## Train / Validation Split

Since the data is time-based, I used a time-based validation split rather than a random split.

I trained the models using data from January to August 2025 and used September and October 2025 as the validation period. This better reflects the real situation of using historical data to predict future loads.


In [14]:
train_data = data[data['date'] < '2025-09-01'].copy()
valid_data = data[data['date'] >= '2025-09-01'].copy()

In [15]:
train_data.shape, valid_data.shape

((38477, 15), (9523, 15))

In [16]:
train_data = fill_missing_weight(train_data, train_data)
valid_data = fill_missing_weight(train_data, valid_data)

In [17]:
print(train_data['weight'].isna().sum())
print(valid_data['weight'].isna().sum())

0
0


## Baseline: Average posted_rate 
Before comparing models, I established a simple baseline using the mean `posted_rate` from the training data.

This provides a reference point for judging whether the machine learning models are actually adding useful predictive value.


In [18]:
baseline_prediction = train_data['posted_rate'].mean()

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_valid = valid_data['posted_rate']

baseline_preds = [baseline_prediction] * len(valid_data)

print('MAE:', mean_absolute_error(y_valid, baseline_preds))
print('RMSE:', mean_squared_error(y_valid, baseline_preds) ** 0.5)
print('R²:', r2_score(y_valid, baseline_preds))

MAE: 1178.8032808646585
RMSE: 1526.2262984310191
R²: -0.00022795590492608042


## Initial Model

After establishing the baseline, I built a first model using the original load features: `distance`, `weight`, `equipment`, `pickup`, and `delivery`.

The categorical features (`equipment`, `pickup`, and `delivery`) were one-hot encoded, while `distance` and `weight` were used as numerical features.

I started with Linear Regression to get a simple model and establish a reference point before doing further feature engineering.


In [20]:
features = [
    'distance',
    'weight',
    'equipment',
    'pickup',
    'delivery'
]

target = 'posted_rate'

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

categorical_features = [
    'equipment',
    'pickup',
    'delivery'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore'
            ),
            categorical_features
        )
    ],
    remainder='passthrough'
)

In [22]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [23]:
X_train = train_data[features]
y_train = train_data[target]

X_valid = valid_data[features]
y_valid = valid_data[target]

In [24]:
model.fit(X_train, y_train)

predictions = model.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, predictions))
print('RMSE:', mean_squared_error(y_valid, predictions) ** 0.5)
print('R²:', r2_score(y_valid, predictions))

MAE: 144.8724561041683
RMSE: 639.8492824412122
R²: 0.8242009604832373


## Feature Engineering

I created `weight_per_distance` to capture the relationship between the weight of a load and the distance it needs to travel.

This produced a small improvement in validation performance, so I kept the feature in the final model.

Date-related features were also tested, but they performed slightly worse on the time-based validation set and were therefore not included in the final model.


In [25]:
train_data['weight_per_distance'] = (
    train_data['weight'] / train_data['distance']
)

valid_data['weight_per_distance'] = (
    valid_data['weight'] / valid_data['distance']
)

In [26]:
features = [
    'distance',
    'weight',
    'weight_per_distance',
    'equipment',
    'pickup',
    'delivery'
]

In [27]:
X_train = train_data[features]
X_valid = valid_data[features]

model.fit(X_train, y_train)

predictions = model.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, predictions))
print('RMSE:', mean_squared_error(y_valid, predictions) ** 0.5)
print('R²:', r2_score(y_valid, predictions))

MAE: 141.65419329794418
RMSE: 638.4759135763587
R²: 0.824954818694967


## Model Comparison

I compared three regression models using the same feature set:

* Linear Regression
* Random Forest
* Gradient Boosting

The models were evaluated using MAE, RMSE and R² on the time-based validation set.

Gradient Boosting produced the lowest validation MAE, so I selected it as the final model.

I also tested a small amount of hyperparameter tuning. Although the tuned model slightly improved RMSE and R², its MAE was slightly worse on the time-based validation set. I therefore kept the simpler Gradient Boosting configuration.


In [28]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        random_state=42
    )
}

In [29]:
results = []

for name, regressor in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_valid)

    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_valid, predictions),
        'RMSE': mean_squared_error(y_valid, predictions) ** 0.5,
        'R²': r2_score(y_valid, predictions)
    })

results = pd.DataFrame(results)

results

,model,MAE,RMSE,R²
0,Linear Regression,141.654193,638.475914,0.824955
1,Random Forest,157.383381,687.759820,0.796888
2,Gradient Boosting,122.276501,637.626372,0.825420


## Final Training and Predictions

After selecting the final model, I retrained it using all of the labelled development data from January to October.

I then used this model to generate predictions for all 12,000 rows in `validation.csv` and saved them as `validation_predictions.csv`.


In [30]:
full_data = fill_missing_weight(data, data)

In [31]:
full_data['weight_per_distance'] = (
    full_data['weight'] / full_data['distance']
)

In [32]:
print(full_data['weight'].isna().sum())
print(full_data['weight_per_distance'].isna().sum())

0
0


In [33]:
X_full = full_data[features]
y_full = full_data[target]

In [34]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=200,
        random_state=42
    ))
])

final_model.fit(X_full, y_full);

### Validation Prediction

In [35]:
validation_data = fill_missing_weight(data, validation.copy())

validation_data['weight_per_distance'] = (
    validation_data['weight'] / validation_data['distance']
)

In [36]:
X_validation = validation_data[features]

In [37]:
validation_data['predicted_rate'] = final_model.predict(X_validation)

C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [38]:
print(validation_data.shape)
print(validation_data['predicted_rate'].isna().sum())

(12000, 15)
0


In [39]:
validation_predictions = validation_data[
    ['load_id', 'predicted_rate']
].copy()

In [40]:
validation_predictions.to_csv(
    'validation_predictions.csv',
    index=False
)

## December Predictions

I also generated the required December predictions using the same final model.

The December input has the same load characteristics for each day. Since `date` was not included as a model feature, the model produces the same predicted rate across the 31 days.

The resulting prediction chart was generated using the provided `score.py`.


In [41]:
december = pd.read_csv('data/december-chart-inputs.csv')

In [42]:
december.isna().sum()

pickup             0
delivery           0
distance           0
equipment          0
weight             0
date               0
predicted_rate    31
dtype: int64

In [43]:
december_data = december.drop(columns='predicted_rate').copy()

In [44]:
december_data['weight_per_distance'] = (
    december_data['weight'] / december_data['distance']
)

In [45]:
december_data['predicted_rate'] = final_model.predict(
    december_data[features]
)

In [46]:
december_predictions = december_data[
    [
        'pickup',
        'delivery',
        'distance',
        'equipment',
        'weight',
        'date',
        'predicted_rate'
    ]
].copy()

In [47]:
december_predictions.to_csv(
    'december_predictions.csv',
    index=False
)

In [49]:
!python score.py --predictions validation_predictions.csv --december-predictions december_predictions.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results\candidate_december.png
Final validation metrics are calculated by Spotter after submission.
